In [23]:
import pandas as pd
import json
from sklearn.metrics import classification_report
from transformers import RobertaModel
from transformers import RobertaTokenizerFast
from transformers import RobertaForSequenceClassification
import torch, torchvision
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset
from torch import nn
from tqdm import tqdm
import sys

In [22]:
# load the data
with open("../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [15]:
class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            for ann in item["annotations"]:
                span_text = ann["text"]
                label = label2id[ann["tag"][3:]]
                # combine sentence and target span
                encoded = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                )
                self.dataset.append({
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                    "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [9]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
split_idx = int(len(data_with_annotations) * 0.75)
train_dataset = data_with_annotations[:split_idx]
test_dataset = data_with_annotations[split_idx:]

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
train_dataset = StanceDataset(train_dataset, tokenizer, label_to_id)
test_dataset = StanceDataset(test_dataset, tokenizer, label_to_id)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)

In [19]:
num_labels = len(label_to_id)
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=num_labels)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    print(f"Average loss: {total_loss / len(train_loader):.4f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Epoch 1/10: 100%|██████████| 47/47 [00:21<00:00,  2.14it/s, loss=0.781]


Average loss: 0.8966


Epoch 2/10: 100%|██████████| 47/47 [00:19<00:00,  2.42it/s, loss=1.3]  


Average loss: 0.7333


Epoch 3/10: 100%|██████████| 47/47 [00:19<00:00,  2.43it/s, loss=0.147]


...

Average loss: 0.0879


Epoch 9/10: 100%|██████████| 47/47 [00:19<00:00,  2.40it/s, loss=0.0139]


Average loss: 0.0780


Epoch 10/10: 100%|██████████| 47/47 [00:19<00:00,  2.39it/s, loss=0.129]  

Average loss: 0.0460


In [20]:
model.eval()
true_labels, pred_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        true_labels.extend(labels.cpu().tolist())
        pred_labels.extend(preds.cpu().tolist())

print(classification_report(
    [list(label_to_id.keys())[i] for i in true_labels],
    [list(label_to_id.keys())[i] for i in pred_labels]
))

              precision    recall  f1-score   support

         neg       0.73      1.00      0.85        11
     neutral       0.66      0.43      0.52        76
         pos       0.79      0.89      0.84       171

    accuracy                           0.76       258
   macro avg       0.73      0.77      0.74       258
weighted avg       0.75      0.76      0.74       258

